# 文をクラスタリングして話題ごとにまとめる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/10_sentence_embeddings/sentence_clustering.ipynb)

NLP クックブック 10-04 ／ 文埋め込みとクラスタリング

大量の文（アンケートの自由記述、問い合わせなど）を、意味の近いもの同士で自動的にグループ分けしたい。

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering, KMeans

texts = [
    "アプリがすぐ落ちる", "起動するとクラッシュします", "ログインできません", "パスワードを忘れてログインできない",
    "料金が高すぎる", "月額プランをもっと安くしてほしい", "解約方法がわからない", "退会の手続きを教えてください",
    "動作が重くて使いにくい", "読み込みが遅い", "値段の割に機能が少ない", "サインインの画面でエラーが出る",
]
model = SentenceTransformer("cl-nagoya/ruri-v3-130m")
# Ruri v3 は、クラスタリングや分類に使うときは先頭に "トピック: " を付ける約束になっている
emb = model.encode(["トピック: " + t for t in texts], normalize_embeddings=True)

In [2]:
# クラスタ数を決め打ちする場合: k-means
labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(emb)
for c in range(4):
    print(c, [t for t, l in zip(texts, labels) if l == c])

0 ['ログインできません', 'パスワードを忘れてログインできない', 'サインインの画面でエラーが出る']
1 ['料金が高すぎる', '月額プランをもっと安くしてほしい', '値段の割に機能が少ない']
2 ['解約方法がわからない', '退会の手続きを教えてください']
3 ['アプリがすぐ落ちる', '起動するとクラッシュします', '動作が重くて使いにくい', '読み込みが遅い']


In [3]:
# クラスタ数がわからない場合: 距離のしきい値で止める階層的クラスタリング
# ward 法はユークリッド距離を使う。長さ 1 に正規化したベクトルなので、コサイン距離と同じ順序になる
agg = AgglomerativeClustering(n_clusters=None, distance_threshold=0.6, linkage="ward")
labels = agg.fit_predict(emb)
for c in np.unique(labels):
    print(c, [t for t, l in zip(texts, labels) if l == c])

0 ['料金が高すぎる', '月額プランをもっと安くしてほしい', '値段の割に機能が少ない']
1 ['アプリがすぐ落ちる', '起動するとクラッシュします']
2 ['動作が重くて使いにくい', '読み込みが遅い']
3 ['解約方法がわからない', '退会の手続きを教えてください']
4 ['ログインできません', 'パスワードを忘れてログインできない', 'サインインの画面でエラーが出る']


- k-means は 4 つの話題（不具合・ログイン・料金・解約）にきれいに分かれる。階層的クラスタリングはクラスタ数を 5 つと決め、不具合を「落ちる・クラッシュ」と「重い・遅い」に分けた。
- 結果はモデルとリンケージ（クラスタ間の距離の測り方）で大きく変わる。`intfloat/multilingual-e5-small` に平均リンケージ (`linkage="average"`) を使うと、クラスタが 3〜6 個になるしきい値では、どれも「解約方法がわからない」がログインの群に、「起動するとクラッシュします」と「サインインの画面でエラーが出る」が同じ群に入った。E5 は類似度が全体に高く出て差が小さいので、こうした短い文の細かい区別には向かない。
- しきい値はモデルとリンケージによって適切な値が違う。少量のデータで目視して決める。`compute_distances=True` を付けて学習すると、`agg.distances_` に結合したときの距離が並ぶので、間が大きく空いたところをしきい値にするとよい。
- 各クラスタの名前付けは、クラスタ内の代表文（中心に最も近い文）を見るか、LLM に要約させるとよい。クラスタの特徴語で名前を付ける方法と、LLM でラベルを付ける方法は、トピックとキーワードの抽出の章の「文埋め込みとクラスタリングでトピックを作る（BERTopic 方式）」と「アンケートの自由記述を話題ごとに集計して報告する（LLM によるラベル付け）」を参照。